# Manuscript benchmark overview

This notebook audits the frozen synthetic manuscript-profile outputs. It reads CSV, JSON, Markdown, and log files from `SCGEO_BENCHMARK_DIR` or the configured default `../scgeo/results/simulation/manuscript_v1`.

No manuscript simulation jobs are run here. Calibration seeds and held-out evaluation seeds are summarized separately, and one simulation job/seed is treated as the independent unit.

## Reader guide

- **Purpose:** Manuscript benchmark overview in the frozen revision workflow.
- **Inference scope:** Controlled synthetic evaluation. One frozen simulation job/seed is the independent unit; no biological claim is made.
- **Inputs:** Checksum-pinned manuscript-profile benchmark tables selected by `SCGEO_BENCHMARK_DIR` and the frozen benchmark config.
- **Implementation:** `scripts/execute_revision_notebooks.py`.
- **Outputs:** Ignored `results/revision_synthetic_benchmark/` review artifacts.
- **Frozen findings:** Report accepted and negative controlled findings, including estimator, representation, local-geometry, dynamics, seed-dependence, recall, and coverage results, without changing thresholds.
- **Limitations:** Synthetic behavior is scenario-specific; controlled corruption sensitivity is not general OOD detection.

This tracked source notebook is intentionally output-free. The clean-kernel runner writes executed review copies and generated artifacts under the ignored results directory.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "scgeo_revision_matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})

SPLIT_COLORS = {"calibration": "#4C78A8", "evaluation": "#F58518"}
METHOD_ORDER = ["shift_mean", "robust_mean", "robust_median", "robust_trimmed_mean", "robust_geometric_median"]
METHOD_LABELS = {
    "shift_mean": "Mean",
    "robust_mean": "Robust mean",
    "robust_median": "Median",
    "robust_trimmed_mean": "Trimmed mean",
    "robust_geometric_median": "Geometric median",
}


def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "configs" / "manuscript_benchmark_v1.json").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "configs" / "manuscript_benchmark_v1.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


def resolve_config_path(value: str) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.resolve()


BENCHMARK_DIR = resolve_config_path(os.environ.get(CONFIG["benchmark_dir_env"], CONFIG["default_benchmark_dir"]))
OUTPUT_DIR = resolve_config_path(os.environ.get(CONFIG["notebook_output_env"], CONFIG["default_output_dir"]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def rel_display(path: Path) -> str:
    path = path.resolve()
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"


def git_commit(path: Path) -> str | None:
    if not (path / ".git").exists():
        return None
    try:
        return subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_manifest() -> pd.DataFrame:
    manifest = pd.read_csv(REPO_ROOT / CONFIG["results_manifest"])
    required_columns = {"relative_path", "role", "size_bytes", "sha256", "protocol_version", "profile", "source_commit"}
    missing = required_columns.difference(manifest.columns)
    if missing:
        raise AssertionError(f"Manifest is missing columns: {sorted(missing)}")
    return manifest


def load_checksum_manifest() -> dict[str, str]:
    checksums = {}
    path = REPO_ROOT / CONFIG["checksums_manifest"]
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        sha, rel = line.split("  ", 1)
        checksums[rel] = sha
    return checksums


def verify_inputs(required_paths: list[str] | None = None, exhaustive_checksums: bool = False) -> dict[str, object]:
    if not BENCHMARK_DIR.exists():
        raise FileNotFoundError(f"Frozen benchmark directory not found: {BENCHMARK_DIR}")

    manifest = load_manifest()
    checksum_manifest = load_checksum_manifest()
    manifest_idx = manifest.set_index("relative_path", drop=False)
    if set(checksum_manifest) != set(manifest_idx.index):
        raise AssertionError("benchmark_files.csv and checksums.sha256 do not describe the same files")
    for rel, expected in checksum_manifest.items():
        actual = manifest_idx.loc[rel, "sha256"]
        if actual != expected:
            raise AssertionError(f"Checksum manifest disagrees with benchmark_files.csv for {rel}")

    if set(manifest["protocol_version"].unique()) != {CONFIG["protocol_version"]}:
        raise AssertionError("Manifest protocol_version does not match config")
    if set(manifest["profile"].unique()) != {CONFIG["profile"]}:
        raise AssertionError("Manifest profile does not match config")
    if set(manifest["source_commit"].unique()) != {CONFIG["expected_source_commit"]}:
        raise AssertionError("Manifest source_commit does not match config")

    selected = set(required_paths or [])
    selected.add("manuscript_jobs.csv")
    selected.update(manifest.loc[manifest["role"].isin(["job_config", "job_status"]), "relative_path"].tolist())
    if exhaustive_checksums:
        selected = set(manifest_idx.index)

    unknown = sorted(selected.difference(manifest_idx.index))
    if unknown:
        raise AssertionError(f"Requested files are not present in the manifest: {unknown[:10]}")

    missing_files = []
    checksum_mismatches = []
    for rel in sorted(selected):
        path = BENCHMARK_DIR / rel
        if not path.exists():
            missing_files.append(rel)
            continue
        expected = manifest_idx.loc[rel, "sha256"]
        actual = sha256_file(path)
        if actual != expected:
            checksum_mismatches.append((rel, expected, actual))
    if missing_files:
        raise FileNotFoundError(f"Missing frozen benchmark files: {missing_files[:10]}")
    if checksum_mismatches:
        first = checksum_mismatches[0]
        raise AssertionError(f"Checksum mismatch for {first[0]}: expected {first[1]}, observed {first[2]}")

    jobs = pd.read_csv(BENCHMARK_DIR / "manuscript_jobs.csv")
    if len(jobs) != CONFIG["expected_jobs"]["total"]:
        raise AssertionError(f"Expected {CONFIG['expected_jobs']['total']} jobs, observed {len(jobs)}")
    if int((jobs["status"] == "completed").sum()) != CONFIG["expected_jobs"]["completed"]:
        raise AssertionError("Not all frozen manuscript jobs are marked completed")

    config_rows = []
    for rel in sorted(manifest.loc[manifest["role"] == "job_config", "relative_path"]):
        data = json.loads((BENCHMARK_DIR / rel).read_text(encoding="utf-8"))
        config_rows.append({
            "job_id": data["job_id"],
            "profile": data["profile"],
            "scenario": data["scenario"],
            "seed_split": data["seed_split"],
            "seed": int(data["seed"]),
            "output_dir": data.get("output_dir", ""),
            "n_boot": data.get("profile_cfg", {}).get("n_boot"),
        })
    job_configs = pd.DataFrame(config_rows)
    if len(job_configs) != CONFIG["expected_jobs"]["total"]:
        raise AssertionError("Job config count does not match expected total")
    if set(job_configs["profile"]) != {CONFIG["profile"]}:
        raise AssertionError("Per-job profile does not match the manuscript profile")
    if sorted(job_configs["scenario"].unique()) != CONFIG["expected_scenarios"]:
        raise AssertionError("Per-job scenarios do not match the revision config")
    if not job_configs["output_dir"].astype(str).str.endswith(CONFIG["protocol_version"]).all():
        raise AssertionError("Per-job output_dir does not record the expected protocol version")

    split_counts = job_configs["seed_split"].value_counts().to_dict()
    for split, expected_count in [("calibration", CONFIG["expected_jobs"]["calibration"]), ("evaluation", CONFIG["expected_jobs"]["evaluation"] )]:
        if int(split_counts.get(split, 0)) != expected_count:
            raise AssertionError(f"Unexpected {split} job count: {split_counts.get(split, 0)}")
    seeds_by_split = job_configs.groupby("seed_split")["seed"].apply(lambda values: sorted(set(map(int, values)))).to_dict()
    for split, expected_seeds in CONFIG["expected_seed_splits"].items():
        if seeds_by_split.get(split) != expected_seeds:
            raise AssertionError(f"Unexpected seeds for {split}: {seeds_by_split.get(split)}")

    source_repo = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
    source_repo_commit = git_commit(source_repo)
    if source_repo_commit is not None and source_repo_commit != CONFIG["expected_source_commit"]:
        raise AssertionError(f"Source repository commit mismatch: {source_repo_commit}")

    return {
        "manifest": manifest,
        "jobs": jobs,
        "job_configs": job_configs,
        "checksum_files_verified": len(selected),
        "source_repo_commit": source_repo_commit,
        "source_repo_commit_verified": source_repo_commit == CONFIG["expected_source_commit"],
        "benchmark_dir": BENCHMARK_DIR,
        "output_dir": OUTPUT_DIR,
    }


def read_audit(filename: str) -> pd.DataFrame:
    return pd.read_csv(BENCHMARK_DIR / "audit" / filename)


def write_table_artifact(stem: str, table: pd.DataFrame, alt_text: str | None = None) -> pd.DataFrame:
    source_dir = OUTPUT_DIR / "figure_sources"
    source_dir.mkdir(parents=True, exist_ok=True)
    table_path = source_dir / f"{stem}.csv"
    table.to_csv(table_path, index=False)
    rows = [{"artifact": stem, "source_csv": rel_display(table_path)}]
    if alt_text is not None:
        alt_dir = OUTPUT_DIR / "alt_text"
        alt_dir.mkdir(parents=True, exist_ok=True)
        alt_path = alt_dir / f"{stem}.txt"
        alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
        rows[0]["alt_text"] = rel_display(alt_path)
    return pd.DataFrame(rows)


def write_figure_bundle(stem: str, fig: plt.Figure, source_table: pd.DataFrame, alt_text: str) -> pd.DataFrame:
    fig_dir = OUTPUT_DIR / "figures"
    source_dir = OUTPUT_DIR / "figure_sources"
    alt_dir = OUTPUT_DIR / "alt_text"
    for directory in (fig_dir, source_dir, alt_dir):
        directory.mkdir(parents=True, exist_ok=True)
    png_path = fig_dir / f"{stem}.png"
    svg_path = fig_dir / f"{stem}.svg"
    csv_path = source_dir / f"{stem}.csv"
    alt_path = alt_dir / f"{stem}.txt"
    source_table.to_csv(csv_path, index=False)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", metadata={"Date": None})
    fig.savefig(png_path, format="png", bbox_inches="tight", metadata={"Software": "matplotlib"})
    alt_path.write_text(alt_text.strip() + "\n", encoding="utf-8")
    plt.close(fig)
    return pd.DataFrame([{ 
        "figure": stem,
        "png": rel_display(png_path),
        "svg": rel_display(svg_path),
        "source_csv": rel_display(csv_path),
        "alt_text": rel_display(alt_path),
    }])


def write_metadata(stem: str, extra: dict[str, object] | None = None) -> pd.DataFrame:
    metadata_dir = OUTPUT_DIR / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    source_repo = resolve_config_path(os.environ.get(CONFIG["source_repository_env"], CONFIG["default_source_repository"]))
    metadata = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": {name: package_version(name) for name in ["scgeo", "pandas", "numpy", "matplotlib", "nbformat", "nbclient"]},
        "notebook_repository_commit": git_commit(REPO_ROOT),
        "source_commit_expected": CONFIG["expected_source_commit"],
        "source_repository_commit": git_commit(source_repo),
        "protocol_version": CONFIG["protocol_version"],
        "profile": CONFIG["profile"],
        "benchmark_dir": str(BENCHMARK_DIR),
        "output_dir": str(OUTPUT_DIR),
        "independent_unit": CONFIG["independent_unit"],
        "threshold_policy": CONFIG["threshold_policy"],
    }
    if extra:
        metadata.update(extra)
    path = metadata_dir / f"{stem}_metadata.json"
    path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return pd.DataFrame([{"metadata": rel_display(path)}])


def split_source_note() -> pd.DataFrame:
    return pd.DataFrame([{
        "calibration_seeds": ",".join(map(str, CONFIG["expected_seed_splits"]["calibration"])),
        "evaluation_seeds": ",".join(map(str, CONFIG["expected_seed_splits"]["evaluation"])),
        "independent_unit": CONFIG["independent_unit"],
        "threshold_policy": CONFIG["threshold_policy"],
    }])

In [ ]:
context = verify_inputs(required_paths=CONFIG["required_audit_files"], exhaustive_checksums=True)
manifest = context["manifest"]
job_configs = context["job_configs"]

integrity = pd.DataFrame([
    {"check": "protocol", "value": CONFIG["protocol_version"]},
    {"check": "source_commit", "value": CONFIG["expected_source_commit"]},
    {"check": "source_repo_commit_verified", "value": context["source_repo_commit_verified"]},
    {"check": "manifest_files", "value": len(manifest)},
    {"check": "checksum_files_verified", "value": context["checksum_files_verified"]},
    {"check": "completed_jobs", "value": int((context["jobs"]["status"] == "completed").sum())},
    {"check": "calibration_jobs", "value": int((job_configs["seed_split"] == "calibration").sum())},
    {"check": "heldout_evaluation_jobs", "value": int((job_configs["seed_split"] == "evaluation").sum())},
])
integrity

In [ ]:
runtime = read_audit("runtime_memory.csv")
overview = runtime[(runtime["scenario"] == "all_scenarios") & runtime["metric"].isin(["runtime_seconds", "peak_memory_mb"])].copy()
overview["metric_label"] = overview["metric"].map({"runtime_seconds": "Runtime (s)", "peak_memory_mb": "Peak memory (MB)"})

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6), constrained_layout=True)
for ax, metric, title in zip(axes, ["runtime_seconds", "peak_memory_mb"], ["Runtime per job", "Recorded peak memory"]):
    subset = overview[overview["metric"] == metric].set_index("seed_split").loc[["calibration", "evaluation"]].reset_index()
    x = np.arange(len(subset))
    ax.bar(x, subset["median"], color=[SPLIT_COLORS[s] for s in subset["seed_split"]], width=0.62)
    lower = subset["median"] - subset["q25"]
    upper = subset["q75"] - subset["median"]
    ax.errorbar(x, subset["median"], yerr=[lower, upper], fmt="none", color="#222222", capsize=4, linewidth=1)
    ax.set_xticks(x, ["Calibration", "Held-out"])
    ax.set_title(title)
    ax.set_ylabel("seconds" if metric == "runtime_seconds" else "MB")
    for i, row in subset.iterrows():
        ax.text(i, row["median"], f"n={int(row['n_jobs'])}", ha="center", va="bottom", fontsize=8)

heldout_runtime = overview[(overview["seed_split"] == "evaluation") & (overview["metric"] == "runtime_seconds")].iloc[0]
heldout_memory = overview[(overview["seed_split"] == "evaluation") & (overview["metric"] == "peak_memory_mb")].iloc[0]
alt = (
    f"Frozen manuscript_v1 benchmark overview with calibration and held-out evaluation shown separately. "
    f"Held-out evaluation includes {int(heldout_runtime['n_jobs'])} jobs, median runtime {heldout_runtime['median']:.1f} seconds, "
    f"and median peak memory {heldout_memory['median']:.1f} MB."
)
artifacts_runtime = write_figure_bundle("00_benchmark_runtime_memory", fig, overview, alt)
artifacts_runtime

In [ ]:
limitations = read_audit("limitations.csv")
negative_components = [
    "balanced_replicate_heterogeneity",
    "state_level_representation_instability",
    "representation_local_distortion",
    "bootstrap_uncertainty_small_effects",
]
negative_results = limitations[limitations["component"].isin(negative_components)].copy()
negative_results = negative_results.sort_values("component")
negative_alt = (
    "Negative-result audit table listing balanced-replicate seed dependence, incomplete state-level representation-instability recall, "
    "zero local-distortion recall in representation-corruption jobs, and imperfect bootstrap uncertainty coverage."
)
artifacts_negative = write_table_artifact("00_negative_results", negative_results, negative_alt)
negative_results

In [ ]:
seed_split_table = (
    job_configs.groupby(["seed_split", "scenario"], as_index=False)
    .agg(n_jobs=("job_id", "nunique"), seeds=("seed", lambda values: ",".join(map(str, sorted(set(values))))))
    .sort_values(["seed_split", "scenario"])
)
write_table_artifact("00_seed_split_by_scenario", seed_split_table)
metadata = write_metadata("00_manuscript_benchmark_overview", {
    "checksum_files_verified": context["checksum_files_verified"],
    "figures": artifacts_runtime["figure"].tolist(),
    "negative_results_table": "00_negative_results",
})
metadata